# Descrição geral
Os dados foram sintetisados usando um script simples, aqui o foco é gerar um conjunto mais simples, mais rapido e mais facil de usar no treinamento

In [ ]:
from geradores import gera_dados_categoria
df_categorias = gera_dados_categoria.run()

In [ ]:
print('Trechos')
print(df_categorias.head())

print('Shape')
print(df_categorias.shape)

print('Potenciais falhas')
print(df_categorias.isnull().sum())

x = df_categorias['descricao']
y = df_categorias['categoria']


# Treinar modelo

Dividir dados em treino e teste

In [ ]:
from sklearn.model_selection import train_test_split

x_treino, x_teste, y_treino, y_teste = train_test_split(
    x, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Tamanho treino:", x_treino.shape)
print("Tamanho teste:", x_teste.shape)

Vetorizar o texto (TF-IDF)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vetorizador = TfidfVectorizer()

# fit_transform no treino (aprende o vocabulário + transforma)
x_treino_vetorizado = vetorizador.fit_transform(x_treino)

# apenas transform no teste (usa o MESMO vocabulário aprendido no treino)
x_teste_vetorizado = vetorizador.transform(x_teste)

print("Formato treino vetorizado:", x_treino_vetorizado.shape)

Treino do modelo

In [ ]:
from sklearn.naive_bayes import MultinomialNB

modelo = MultinomialNB()
modelo.fit(x_treino_vetorizado, y_treino)

Avaliacao

In [ ]:
from sklearn.metrics import classification_report

y_previsto = modelo.predict(x_teste_vetorizado)

print(classification_report(y_teste, y_previsto))

Teste com novos itens

In [ ]:
exemplos_novos = [
    "Compra de ações",
    "supermerc",
    "uber pra faculdade",
    "netflix mensal",
    "financiamento carro parcela",
    "pizzaria ",
    "metrô"
]

exemplos_vetorizados = vetorizador.transform(exemplos_novos)
previsoes = modelo.predict(exemplos_vetorizados)

for texto, categoria in zip(exemplos_novos, previsoes):
    print(f"'{texto}' -> {categoria}")

# exportar modelo

In [ ]:
import joblib

joblib.dump(vetorizador, "../servico-dados/modelos/vetorizador_categoria.joblib")
joblib.dump(modelo, "../servico-dados/modelos/modelo_categoria.joblib")